<a href="https://colab.research.google.com/github/customerdelightgroup/customerdelight/blob/GroupCode/v2_delighting_the_customer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import nltk
import re
from collections import Counter

nltk.download("stopwords")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import pandas as pd

# load all three datasets
amazon = pd.read_csv("amazon_cells_labelled.txt", sep="\t", header=None, names=["sentence", "label"])
imdb = pd.read_csv("imdb_labelled.txt", sep="\t", header=None, names=["sentence", "label"])
yelp = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["sentence", "label"])

# combine them into one DataFrame
df = pd.concat([amazon, imdb, yelp], ignore_index=True)

df.head()


,sentence,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [ ]:
# cleans sentence so model can understand better
def clean_text(text):
    text = text.lower()                                # make all letters lowercase
    text = re.sub(r"[^a-zA-Z\s]", "", text)            # remove punctuation and special characters
    text = re.sub(r"\s+", " ", text).strip()           # remove extra spaces
    return text

# apply cleaning to every sentence in the dataset
df["clean_sentence"] = df["sentence"].apply(clean_text)

# show first few cleaned sentences
df[["sentence", "clean_sentence"]].head()


,sentence,clean_sentence
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great


In [ ]:
from nltk.corpus import stopwords

# load list of common English stopwords (words like "the", "is", "and")
stop_words = set(stopwords.words("english"))

# remove "no" and "not" from the stopword list because they affect sentiment
stop_words = {w for w in stop_words if w not in ["no", "not"]}

# splits each cleaned sentence into individual words
def tokenize(sentence):
    tokens = sentence.split()                          # split the sentence into words
    tokens = [t for t in tokens if t not in stop_words] # remove stopwords
    return tokens

# apply tokenization to every cleaned sentence
df["tokens"] = df["clean_sentence"].apply(tokenize)

# show first few tokenized sentences
df[["clean_sentence", "tokens"]].head()


,clean_sentence,tokens
0,so there is no way for me to plug it in here i...,"[no, way, plug, us, unless, go, converter]"
1,good case excellent value,"[good, case, excellent, value]"
2,great for the jawbone,"[great, jawbone]"
3,tied to charger for conversations lasting more...,"[tied, charger, conversations, lasting, minute..."
4,the mic is great,"[mic, great]"


In [ ]:
# count words in each token list
df["word_count"] = df["tokens"].apply(len)

# show  first few word counts
df[["tokens", "word_count"]].head()


,tokens,word_count
0,"[no, way, plug, us, unless, go, converter]",7
1,"[good, case, excellent, value]",4
2,"[great, jawbone]",2
3,"[tied, charger, conversations, lasting, minute...",6
4,"[mic, great]",2


In [ ]:
positive_words = Counter()   # store words from positive sentences
negative_words = Counter()   # store words from negative sentences

# scan each row in dataset
for _, row in df.iterrows():
    if row["label"] == 1:                     # if the sentence is positive
        positive_words.update(row["tokens"])  # add its words to the positive counter
    else:
        negative_words.update(row["tokens"])  # otherwise add to the negative counter

# show top 10 most common words in each group
positive_words.most_common(10), negative_words.most_common(10)


([('great', 190),
  ('good', 172),
  ('phone', 86),
  ('movie', 84),
  ('film', 84),
  ('one', 68),
  ('really', 60),
  ('food', 60),
  ('best', 59),
  ('like', 58)],
 [('not', 252),
  ('movie', 93),
  ('bad', 92),
  ('phone', 76),
  ('one', 75),
  ('film', 71),
  ('dont', 66),
  ('like', 65),
  ('food', 64),
  ('no', 62)])

In [ ]:
from nltk import bigrams

# create bigrams for each sentence
def get_bigrams(tokens):
    return list(bigrams(tokens))

df["bigrams"] = df["tokens"].apply(get_bigrams)

# count all bigrams
all_bigrams = Counter()
for bg_list in df["bigrams"]:
    all_bigrams.update(bg_list)

# show 10 most common bigrams
all_bigrams.most_common(10)


[(('not', 'good'), 22),
 (('go', 'back'), 18),
 (('works', 'great'), 17),
 (('waste', 'time'), 17),
 (('would', 'not'), 16),
 (('customer', 'service'), 14),
 (('ive', 'ever'), 14),
 (('sound', 'quality'), 13),
 (('could', 'not'), 13),
 (('one', 'best'), 13)]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF turns text into numbers based on how importance
vectorizer = TfidfVectorizer()

# fit vectorizer on cleaned sentences and transform into a matrix
X = vectorizer.fit_transform(df["clean_sentence"])

# our labels (0 = negative, 1 = positive)
y = df["label"]


In [ ]:
from sklearn.model_selection import train_test_split

# split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


((2198, 5283), (550, 5283))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# create the Naive Bayes model
nb_model = MultinomialNB()

# train the model
nb_model.fit(X_train, y_train)

# make predictions on the test set
nb_predictions = nb_model.predict(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression

# create Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)

# train the model
lr_model.fit(X_train, y_train)

# make predictions on the test set
lr_predictions = lr_model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# evaluate Naive Bayes
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_predictions))
print("Naive Bayes Precision:", precision_score(y_test, nb_predictions))
print("Naive Bayes Recall:", recall_score(y_test, nb_predictions))
print("Naive Bayes Confusion Matrix:\n", confusion_matrix(y_test, nb_predictions))

# evaluate Logistic Regression
print("\nLogistic Regression Accuracy:", accuracy_score(y_test, lr_predictions))
print("Logistic Regression Precision:", precision_score(y_test, lr_predictions))
print("Logistic Regression Recall:", recall_score(y_test, lr_predictions))
print("Logistic Regression Confusion Matrix:\n", confusion_matrix(y_test, lr_predictions))


Naive Bayes Accuracy: 0.8163636363636364
Naive Bayes Precision: 0.7862318840579711
Naive Bayes Recall: 0.8378378378378378
Naive Bayes Confusion Matrix:
 [[232  59]
 [ 42 217]]

Logistic Regression Accuracy: 0.8163636363636364
Logistic Regression Precision: 0.8038461538461539
Logistic Regression Recall: 0.806949806949807
Logistic Regression Confusion Matrix:
 [[240  51]
 [ 50 209]]


Naive Bayes
Confusion Matrix Results:
232 true negatives
217 true positives
59 false positives
42 false negatives

Conclusion: Naive Bayes is slightly better at recall, meaning it catches more positive reviews, but it makes more false positives. Therefore, better at recall - catching more positives.

Logistic Regression
Confusion Matrix Results:
240 true negatives
209 true positives
51 false positives
50 false negatives

Conclusion:  Logistic Regression is more balanced — less false positives, slightly more false negatives. Therefore, better at precisions - fewer mistakes.

Both models have the same accuracy (0.816)